# # 02 · From m/z peaks to lipid names

Companion to the tutorial notebook [`02_annotation`](https://github.com/lamanno-epfl/lipidomics_tutorial_cajalcourse/tree/main/notebooks/level1).

**Kernel:** `cajal-lipidomics` · **Reads:** `01_raw.h5ad` + LC-MS / LIPID MAPS references · **Writes:** `data/derived/02_annotated.h5ad`

The instrument hands us masses like `518.2643`, never identities. This notebook turns each measured peak into a lipid name we can defend. That's **annotation**, and we'll be honest about how much to trust each name.

**The plan**
1. A peak is a mass, not a lipid
2. Adducts and de-ionization (`neutral = observed − adduct`)
3. The ppm formula: how close is close enough?
4. Matching against an LC-MS reference, and seeing the match
5. Isobaric ties, and breaking them with bulk abundance (the 80% rule)
6. The database layer: LIPID MAPS
7. Annotating every ion, reading names into properties, saving

In [ ]:
# ---- setup: the same first cell in every notebook ---------------------------------------

# My own little package. `paths` knows where the tutorial clone, its provided data,
# and my own outputs live, so I never type "../../data/..." by hand.
from lipid_project import paths

paths.ensure_dirs()   # creates data/derived/, figures/, results/ if they're missing (safe to re-run)
paths.summary()       # prints where everything points + which stages are built so far.

import re                          # regular expressions, for reading formulas and lipid names
import numpy as np                 # arrays and math
import pandas as pd                # tables
import matplotlib.pyplot as plt    # plotting
import anndata as ad               # the AnnData container

# The tutorial's helper package. Read annotation.py before using it: it's ~70 short lines.
import cajal_lipidomics as cl
from cajal_lipidomics import annotation as cann, plotting    # cann = the tutorial's annotation helpers (adduct table, ppm maths, LC-MS loader)
from cajal_lipidomics.style import FS
cl.style.set_style()    # the tutorial's matplotlib style, so my figures match theirs

REFS = paths.REFS / "csv"          # the LC-MS reference tables live in <tutorial>/data/refs/csv/
PPM_TOL = 5.0                      # the matching window, same as the paper

print("ready. cajal_lipidomics", cl.__version__)

# ## 1. A peak is a mass, not a lipid

We load notebook 01's output. Each column is an ion keyed by `formula__adduct` with a measured `mz`. METASPACE already proposed a formula, but a formula isn't a lipid name, and one formula can hide several different lipids.

In [ ]:
raw = ad.read_h5ad(paths.stage("raw"))    # the pixels x ions table that notebook 01 saved
print("raw substrate:", raw.shape, "(pixels x ions)")
print("conditions:", raw.obs["Condition"].value_counts().to_dict())
raw.var.head(8)

In [ ]:
# The mean spectrum of the whole dataset: average each ion over all pixels, one stick per ion
mz_vals = raw.var["mz"].to_numpy(float)    # one m/z per ion (column)
mean_int = np.asarray(raw.X).mean(axis=0)    # average intensity of each ion over all pixels = a 'mean spectrum'
ax = plotting.spectrum(mz_vals, mean_int, title="mean MALDI spectrum of the raw ions (each stick = one peak)")
plt.show()
print(f"{len(mz_vals)} peaks spanning m/z {mz_vals.min():.1f} to {mz_vals.max():.1f}")

# Every stick is a mass the instrument measured to several decimals, and not one is labelled with a molecule. The instrument resolves **mass**, never **identity**.

# ## 2. Adducts and de-ionization

In positive mode a lipid flies as an **ion**: its neutral mass plus a small attached ion, the **adduct**.

| adduct | what attaches | adds (Da) |
|---|---|---|
| `[M+H]+` | a proton | 1.007276 |
| `[M+Na]+` | sodium | 22.989769 |
| `[M+K]+` | potassium | 38.963707 |
| `[M+NH4]+` | ammonium | 18.033823 |

Reference databases store the **neutral** mass (a property of the molecule itself). So to compare, we undo the ionization: **`neutral = observed − adduct`**.

To make this concrete we compute a neutral mass straight from a molecular formula. The **monoisotopic mass** is the sum of each atom's most common isotope mass.

In [ ]:
# Exact mass of each element's most abundant isotope (Da)
ATOM = {"C": 12.000000, "H": 1.00782503207, "N": 14.0030740048, "O": 15.99491461956,
        "P": 30.97376163, "S": 31.97207100, "Na": 22.98976928, "K": 38.96370668, "Fe": 55.9349375}


def neutral_mass(formula):
    """Monoisotopic mass of a formula like 'C42H82NO8P'.

    The regex finds (element, count) pairs: an uppercase letter, an optional lowercase
    letter (so 'Na' and 'Fe' work), then optional digits. A missing count means 1.
    """
    # add up atom mass x atom count
    return sum(ATOM[el] * (int(n) if n else 1) for el, n in re.findall(r"([A-Z][a-z]?)(\d*)", formula))


# PC 34:1 has formula C42H82NO8P. Where would it appear under each adduct?
pc = neutral_mass("C42H82NO8P")    # PC 34:1, one of the most abundant lipids in brain
print(f"PC 34:1 neutral mass = {pc:.4f} Da, so it can show up at:")
for name, shift in cann.ADDUCTS.items():
    print(f"   [M+{name[:-1]}]+  ->  m/z {pc + shift:9.4f}")

# And which adducts did our dataset actually use?
print("\nadducts in our raw ions:")
print(raw.var["adduct"].value_counts().to_string())

# Potassium adducts dominate in brain tissue (there's a lot of K+ around). One lipid can show up at four masses, which is why a raw peak list always has more peaks than lipids.

Now de-ionization on a real peak from our data. We pick the first ion as the worked example, then try all four adduct hypotheses.

In [ ]:
observed = float(raw.var["mz"].iloc[0])    # take the first ion as a worked example
print(f"worked peak: m/z {observed:.4f}  (METASPACE says adduct {raw.var['adduct'].iloc[0]}, "
      f"formula {raw.var['formula'].iloc[0]})")
for name, shift in cann.ADDUCTS.items():
    print(f"   if it is [M+{name[:-1]}]+ : neutral = {observed:.4f} - {shift:7.4f} = {observed - shift:9.4f} Da")

# Sanity check: METASPACE's own formula + adduct should reproduce the observed mass
f0, a0 = raw.var["formula"].iloc[0], raw.var["adduct"].iloc[0]    # what METASPACE claims this peak is
predicted = neutral_mass(f0) + cann.ADDUCTS[a0.lstrip("+") + "+"]    # forward check: formula mass + adduct shift should land on the observed m/z
print(f"\nformula {f0} + adduct {a0} predicts m/z {predicted:.4f} (observed {observed:.4f})")

# ## 3. The ppm formula: how close is close enough?

Measured and reference masses are never exactly equal. We call it a match if they agree **within a tolerance**, measured in **parts per million** because a fixed error in Daltons means different things at mass 200 and mass 1000:

**ppm error = 10⁶ × |observed − reference| / reference**

We use **5 ppm**, like the paper. Since it's relative, it's a wider window in Daltons at higher mass.

In [ ]:
# ppm is relative, so the same 5 ppm is a wider window (in Da) at higher m/z
for m in (200, 800, 1000):
    da = PPM_TOL / 1e6 * m    # ppm -> Da: tolerance x 1e-6 x m/z
    print(f"5 ppm at m/z {m:4d}  =  a {da:.4f} Da window  (+/- {da / 2 * 1000:.1f} mDa)")

# cann.ppm_error is literally that one formula; check it on a known case
print("\na peak 0.004 Da off at m/z 800 is", f"{cann.ppm_error(800.004, 800.0):.2f} ppm away")

# ## 4. Matching against an LC-MS reference

The strongest reference is an independent measurement: **LC-MS** on similar brain tissue. Liquid chromatography separates lipids in time before weighing them, and a tandem step (LC-MS/MS) fragments them to read their structure. The result is a list of confidently identified lipids with their **ionized** m/z, so we can compare MSI peaks to it directly.

First by hand (three lines), then the helper, then a check that they agree.

In [ ]:
# the lab's LC-MS list: every lipid actually measured in brain, with its adduct m/z values
lcms = cann.load_lcms_reference(REFS / "lcms_mar2022_withcounterions (2).txt")
print("LC-MS reference:", lcms.shape, "(lipid, adduct, m/z) rows")
print(lcms.head(6).to_string(index=False))

# By hand: the ppm error from our peak to EVERY reference row, then keep the ones within 5 ppm
ppm_to_all = cann.ppm_error(observed, lcms["m/z"].to_numpy(float))    # distance in ppm from my peak to EVERY reference row, vectorised
inside = ppm_to_all <= PPM_TOL    # boolean mask: rows within the tolerance window
hits_manual = lcms.loc[inside].assign(ppm=ppm_to_all[inside]).sort_values("ppm")    # keep those rows, add their ppm, closest first
print(f"\nreference rows within {PPM_TOL} ppm of m/z {observed:.4f}:")
print(hits_manual.to_string(index=False))

# The helper does exactly those three lines
hits = cann.match_lcms(observed, lcms, ppm_tol=PPM_TOL)    # the tutorial's one-liner for the same thing
assert set(hits["Lipid"]) == set(hits_manual["Lipid"]), "helper and hand-rolled matcher should agree"
print("\nhelper and hand-rolled matcher agree.")

# The helper `plot_ppm_match` makes the match visible: LC-MS reference sticks on top, our MSI peak below, a gold band for ±5 ppm. Green labels fall inside the window (they could donate their name), grey ones are too far.

In [ ]:
cann.plot_ppm_match(observed, lcms, ppm_tol=PPM_TOL, span_ppm=40)
plt.tight_layout()
plt.show()

# ## 5. Isobaric ties, and how to break them

If two *different* lipids land inside the window, that's an **isobaric tie**: different molecules, same mass. It usually happens because they share a formula (e.g. LPC 15:1 and LPE 18:1 are both C23H46NO7P), so no amount of mass precision can separate them.

Biology can, though. A **quantitative bulk LC-MS** run reports how much of each lipid is actually in brain (nmol % of total lipid). The paper's rule: among a peak's candidates, normalise their bulk abundances to fractions, and if the top candidate holds **more than 80%**, keep it. If no candidate is that dominant, the tie stays honest and unresolved (we fall back to the closest-ppm hit and flag it).

First, a small function to tidy LC-MS shorthand into the course's sum-composition style (`SM(d36:1)` → `SM 36:1`, `PC 34.1` → `PC 34:1`), so names from different tables can be compared.

In [ ]:
def clean_name(nm):
    """Tidy LC-MS shorthand into 'CLASS C:DB' style, e.g. 'HexCer(d42:2)' -> 'HexCer 42:2'."""
    nm = str(nm).replace("(d", " ").replace("(", " ").replace(")", "")    # 'HexCer(d42:2)' -> 'HexCer 42:2' (the d = sphingoid base, dropped)
    # Only swap a "." for ":" when it sits between two numbers (34.1 -> 34:1).
    # A blind replace would also turn decimals elsewhere into colons.
    nm = re.sub(r"(\d+)\.(\d+)", r"\1:\2", nm)    # some rows write 34.1 instead of 34:1
    return re.sub(r"\s+", " ", nm).strip()            # collapse any doubled spaces


for nm in ["SM(d36:1)", "HexCer(d42:2)", "PC 34.1", "PE 40:6"]:
    print(f"{nm:15s} -> {clean_name(nm)}")

In [ ]:
# The bulk quantitative LC-MS: rows are lipids (LIPID_ID), columns are samples.
# quantitative bulk LC-MS: how much of each lipid there is in whole brain. Rows without an ID are dropped
qlcms = pd.read_csv(REFS / "QuantitativeLCMS.csv").dropna(subset=["LIPID_ID"]).copy()
female_cols = [c for c in qlcms.columns if c.startswith("Female")]   # our sections are female brains
qlcms[female_cols] = qlcms[female_cols].apply(pd.to_numeric, errors="coerce")    # some cells are text; coerce turns them into NaN instead of crashing
# Mean over the female samples -> one bulk abundance per lipid name
# clean the names, average duplicates per name, then average across the female samples -> one number per lipid
bulk = qlcms.assign(name=qlcms["LIPID_ID"].map(clean_name)).groupby("name")[female_cols].mean().mean(axis=1)
print(f"bulk abundances for {len(bulk)} lipids, from {len(female_cols)} female samples")


def resolve(hits, bulk, share=0.8):
    """Pick one name for a peak from its LC-MS hits.

    Returns (name, ppm, n_candidates, rule). `rule` says how the name was chosen:
      'unique'      only one distinct lipid in the window
      'abundance'   several, but one holds > `share` of their summed bulk abundance
      'ppm (tie)'   several and no clear winner: we keep the closest by ppm, flagged
      'none'        nothing within the window
    """
    if len(hits) == 0:
        return None, np.nan, 0, "none"
    h = hits.assign(name=hits["Lipid"].map(clean_name))    # several adduct rows can be the same lipid, so collapse on the cleaned name
    best = h.groupby("name")["ppm"].min().sort_values()      # one row per distinct lipid, closest first
    if len(best) == 1:    # rule 1: only one lipid fits, done
        return best.index[0], best.iloc[0], 1, "unique"
    ab = bulk.reindex(best.index).fillna(0.0)                 # not measured in bulk brain -> 0
    if ab.sum() > 0 and ab.max() / ab.sum() > share:    # rule 2: one candidate dominates bulk abundance (>80%), so the peak is almost certainly mostly that one
        winner = ab.idxmax()
        return winner, best[winner], len(best), "abundance"
    return best.index[0], best.iloc[0], len(best), "ppm (tie)"    # rule 3: genuine tie, keep the closest by ppm and flag it


# Find a worked example of a tie in OUR data: the first ion whose window holds 2+ distinct lipids
tie_ion = None    # scan the ions until I find one where 2+ different lipids fit, to demo the tie-breaker
for k, mz in raw.var["mz"].items():
    h = cann.match_lcms(float(mz), lcms, ppm_tol=PPM_TOL)
    if h["Lipid"].map(clean_name).nunique() > 1:    # more than one distinct lipid in the window = an isobaric tie
        tie_ion, tie_hits = k, h
        break    # one example is enough

if tie_ion is None:
    print("no isobaric tie among these ions (unusual for brain data)")
else:
    print(f"tie at {tie_ion} (m/z {raw.var.loc[tie_ion, 'mz']:.4f}):")
    print(tie_hits.assign(name=tie_hits["Lipid"].map(clean_name))[["name", "Adduct", "ppm"]].to_string(index=False))
    cands = tie_hits["Lipid"].map(clean_name).unique()
    print("\nbulk brain abundance of each candidate (nmol %; NaN = never measured in brain):")
    print(bulk.reindex(cands).to_string())
    print("\nresolved as:", resolve(tie_hits, bulk))

# When one candidate is abundant in brain and its rival is essentially absent (for example, odd-chain lyso-PCs like LPC 15:1 are vanishingly rare in mammals), the abundant one almost certainly made the peak. When both are plausible, we keep the tie flagged instead of pretending to know.

> Note: the tutorial deliberately keeps the simpler "closest ppm wins" rule. Here we apply the paper's 80% abundance rule on top, so a few of our names may differ from the tutorial's. Both choices are written in code you can read.

# ## 6. The database layer: LIPID MAPS

Many peaks have no LC-MS hit. For those, the fallback is a chemical database: **LIPID MAPS** (`structures.sdf`) stores every known lipid structure with its neutral exact mass and an abbreviation. Matching is the same idea: de-ionize under each adduct, then ppm-match against `EXACT_MASS`.

Key lesson: a database lists *every* known structure, rare ones included, so it can confirm a mass exists but **can't break isobaric ties**. Only the bulk LC-MS knows what's actually abundant in brain.

In [ ]:
from rdkit import Chem, RDLogger    # rdkit reads chemistry files like the LIPID MAPS .sdf
RDLogger.DisableLog("rdApp.*")          # RDKit prints a warning for every odd structure; silence it

# SDMolSupplier reads the .sdf one molecule at a time. We only need three text properties
# per record, so we keep those and skip anything without a mass or abbreviation.
records = []
for mol in Chem.SDMolSupplier(str(paths.REFS / "structures.sdf")):    # iterates over every molecule in the file, one at a time
    if mol is None:    # rdkit returns None for entries it can't parse; just skip them
        continue
    p = mol.GetPropsAsDict()    # the text fields stored with each structure (name, mass, formula...)
    if "EXACT_MASS" in p and "ABBREVIATION" in p:
        records.append((p["ABBREVIATION"], float(p["EXACT_MASS"]), p.get("FORMULA", "")))
lipidmaps = pd.DataFrame(records, columns=["ABBREVIATION", "EXACT_MASS", "FORMULA"])
print(f"parsed {len(lipidmaps):,} LIPID MAPS structures")
print(lipidmaps.head(5).to_string(index=False))

In [ ]:
def lipidmaps_hits(mz, db, ppm_tol=PPM_TOL):
    """All LIPID MAPS structures within ppm_tol of mz under any adduct hypothesis."""
    out = []
    masses = db["EXACT_MASS"].to_numpy()    # LIPID MAPS stores neutral masses
    for adduct, shift in cann.ADDUCTS.items():
        neutral = mz - shift                                  # de-ionize under this hypothesis
        e = cann.ppm_error(neutral, masses)    # compare my peak's implied neutral mass against all structures at once
        for j in np.where(e <= ppm_tol)[0]:
            out.append((db["ABBREVIATION"].iloc[j], adduct, round(float(e[j]), 2)))
    return pd.DataFrame(out, columns=["ABBREVIATION", "adduct", "ppm"]).sort_values("ppm")


example = tie_ion if tie_ion is not None else raw.var_names[0]    # use the tie ion if I found one, it's the more interesting case
print(f"LIPID MAPS candidates for {example} (m/z {raw.var.loc[example, 'mz']:.4f}):")
print(lipidmaps_hits(float(raw.var.loc[example, "mz"]), lipidmaps).to_string(index=False))

# The full paper pipeline matches each peak against several references at once (two LC-MS runs, an LC-MS/MS run, METASPACE/HMDB, LIPID MAPS, published datasets) and adds up confidence weights (structural LC-MS/MS counts most). We keep the transparent single-reference version.

# ## 7. Annotate every ion

Same matcher, applied to all ions: LC-MS hits within 5 ppm, ties resolved with the 80% rule, and a placeholder `ion_<mz>` when nothing matches. We also store *how* each name was chosen, so later notebooks (and you) can see which names are solid.

In [ ]:
rows = []    # now annotate EVERY ion with the same match + resolve logic
for k, mz in raw.var["mz"].astype(float).items():
    name, ppm, n_cand, rule = resolve(cann.match_lcms(mz, lcms, ppm_tol=PPM_TOL), bulk)
    rows.append((name if name else f"ion_{mz:.3f}", ppm, n_cand, rule))    # unmatched peaks keep a placeholder name so nothing gets dropped

ann = pd.DataFrame(rows, columns=["lipid", "ppm", "n_candidates", "annotation_rule"], index=raw.var_names)
raw.var = raw.var.join(ann)                                 # add the new columns next to formula/adduct/mz

annotated = ~raw.var["lipid"].str.startswith("ion_")    # True for ions that got a real lipid name
print(f"annotated {int(annotated.sum())} / {raw.n_vars} ions within {PPM_TOL} ppm of an LC-MS lipid")
print("\nhow each name was chosen:")
print(raw.var["annotation_rule"].value_counts().to_string())
raw.var[["mz", "adduct", "lipid", "ppm", "annotation_rule"]].head(12)

In [ ]:
# Which lipid classes did we recover? The class is the text before the first space.
classes = raw.var.loc[annotated, "lipid"].str.split(" ").str[0]    # class = everything before the first space ('PC 34:1' -> 'PC')
print("annotated lipids per class:")
print(classes.value_counts().to_string())

# ### Reading names into properties

A name is a compressed sentence: class, total carbons, double bonds, plus an **ether** flag (`O-` or `P-` marks an ether-linked chain). We parse every annotated name into a property table. These are sum compositions, so each name already carries one `C:DB` total.

In [ ]:
def parse_lipid(name):
    """(class, carbons, double_bonds, ether) from a sum-composition name like 'PE O-38:7'."""
    lipid_class = name.split(" ")[0]    # e.g. 'PE'
    ether = bool(re.search(r"\b[OP]-", name))           # "O-" or "P-" right before the chain
    m = re.search(r"(\d+):(\d+)", name)    # grab carbons:double-bonds, e.g. 38:7
    return lipid_class, (int(m.group(1)) if m else None), (int(m.group(2)) if m else None), ether


props = pd.DataFrame([parse_lipid(n) for n in raw.var.loc[annotated, "lipid"]],
                     columns=["class", "carbons", "double_bonds", "ether"],
                     index=raw.var_names[annotated])
props["mz"] = raw.var.loc[annotated, "mz"].astype(float)    # keep m/z too, used for marker size below
print("property table:", props.shape)
props.head(8)

In [ ]:
# The property landscape: total carbons vs double bonds, one colour per class
fig, ax = plt.subplots(figsize=(7.5, 5))
order = props["class"].value_counts().index    # biggest class first, so the legend reads sensibly
for col, c in zip(plotting.distinct_colors(len(order)), order):
    sub = props[props["class"] == c]
    ax.scatter(sub["carbons"], sub["double_bonds"], s=18 + sub["mz"] / 20, color=col, alpha=0.8,
               label=c, edgecolor="none")
ax.set_xlabel("total acyl carbons")
ax.set_ylabel("double bonds (unsaturation)")
ax.set_title("annotated lipids by class, carbons and unsaturation (marker size ~ m/z)")
ax.legend(ncol=3, fontsize=FS["xs"], loc="upper left")
plt.tight_layout()
plt.show()

# Lyso-lipids (one chain) sit at low carbon counts; diacyl phospholipids spread to 40+. Losing a fatty tail moves a lipid left on this plot.

### The proof that annotation is real: a named lipid draws anatomy

We map our worked peak, now with its name. A real lipid paints coherent territories; noise wouldn't.

In [ ]:
named = raw.copy()    # a copy with lipid names as var_names, only for plotting
# Use lipid names as column labels. make_index_unique adds "-1", "-2" when two ions share a name
# (e.g. the same lipid detected as [M+H]+ and [M+K]+).
named.var_names = ad.utils.make_index_unique(pd.Index(raw.var["lipid"].astype(str)))    # two peaks can get the same name (e.g. H+ and Na+ of one lipid); this appends -1, -2
show = named.var_names[list(raw.var_names).index(example)]    # the new name of my worked-example ion

plotting.spatial_lipid(named, show)
plt.suptitle(f"{show}  (m/z {raw.var.loc[example, 'mz']:.4f}, finally named and mapped)", y=1.02, fontsize=FS["m"])
plt.show()

# ### Save the annotated stage

Same matrix, same pixels; `var` now also carries `lipid`, `ppm`, `n_candidates` and `annotation_rule`.

In [ ]:
raw.write_h5ad(paths.stage("annotated"))        # -> data/derived/02_annotated.h5ad
print("saved", paths.stage("annotated"))
print("var columns:", list(raw.var.columns))

# ## Recap

- A peak is a **mass**; annotation turns it into a **name** within a ppm window (5 ppm ≈ 0.004 Da at m/z 800).
- Adducts spread one lipid over several masses; de-ionization (`neutral = observed − adduct`) lets us compare to databases.
- **Isobaric ties** can't be broken by mass. Bulk abundance can: the 80% rule picks a winner only when the evidence is clear.
- Databases give breadth, LC-MS gives confidence.
- Saved **`data/derived/02_annotated.h5ad`**, with every name's provenance recorded.

**Next (03):** make the two sections comparable with uMAIA.